In [1]:
# Cell 1: Environment Setup

include("../FullWorkflow/scripts/dictionaries.jl")
include("../FullWorkflow/scripts/helpers.jl")
using .FullWorkflowHelpers
using OMJulia

# --- Configuration ---

# 1. Root model to build (qualified package name)
MODEL = "MyIEEE14.IEEE14DisconnectLine"

# 2. Directory containing the source package
SOURCE_PACKAGE = split(MODEL, ".")[1]
MODEL_DIR = abspath(SOURCE_PACKAGE)
MODELS_PKG_PATH = joinpath(MODEL_DIR, "package.mo")

# 3. Path to the Dynawo package.mo
DYNAWO_PKG_PATH = "/home/clarafercas/Model_library/dynawo/dynawo/sources/Models/Modelica/Dynawo/package.mo"

# 4. Path to the Modelica package.mo
MODELICA_PKG_PATH = "/home/clarafercas/dynawo/OpenModelica/lib/omlibrary/Modelica/package.mo"

# 5. INIT model selection for components with multiple INIT profiles (leave empty for default).
INIT_MODEL_BY_COMPONENT = Dict{String, String}()

# 6. Slack component (leave empty to disable slack-specific handling).
SLACK_COMPONENT = "Gen1"

"Gen1"

In [2]:
# Cell 2: OpenModelica Setup + User's Case Validation

# 1. Start OMC and load libraries
omc = OMJulia.OMCSession()
omc_call(omc, "loadFile(\"$MODELICA_PKG_PATH\")")
omc_call(omc, "loadModel(Complex)")
omc_call(omc, "loadModel(ModelicaServices)")
omc_call(omc, "loadFile(\"$DYNAWO_PKG_PATH\")")

# 2. Load the user's case and validate the configuration against it
omc_call(omc, "loadFile(\"$MODELS_PKG_PATH\")")
config = check_user_configuration_package(omc;
    model = MODEL,
    slack_component = SLACK_COMPONENT,
    init_model_by_component = INIT_MODEL_BY_COMPONENT,
)
chain = config.model_chain

[ Info: Path to zmq file="/tmp/openmodelica.clarafercas.port.julia.EqXBFmkRsb"


Configuration checked successfully


2-element Vector{String}:
 "MyIEEE14.IEEE14Base"
 "MyIEEE14.IEEE14DisconnectLine"

In [3]:
# Cell 3: Auxiliary Package Naming and Paths
PATHS = package_workflow_paths(MODEL, MODEL_DIR, abspath("."))

AUX_PACKAGE = PATHS.aux_package
AUX_DIR = PATHS.aux_dir
AUX_ROOT_MODEL = PATHS.aux_root_model
AUX_PACKAGE_FILE = PATHS.aux_package_file
AUX_ORDER_FILE = PATHS.aux_order_file

AUX_NAME_MAP = auxiliary_name_map(chain, AUX_PACKAGE)

Dict{String, String} with 2 entries:
  "MyIEEE14.IEEE14Base"           => "MyIEEE14_auxiliary.IEEE14Base_auxiliary"
  "MyIEEE14.IEEE14DisconnectLine" => "MyIEEE14_auxiliary.IEEE14DisconnectLine_a…

In [4]:
# Cell 4: Main Auxiliary Build Pipeline

package_context = collect_package_component_contexts(omc, chain)
components_by_model = package_context.components_by_model
patch_components_by_model = package_context.patch_components_by_model
global_blacklist_names = package_context.global_blacklist_names

# Create an empty auxiliary package in OpenModelica
sendExpression(omc, "deleteClass($AUX_PACKAGE)")
omc_call(omc, "loadString(\"within ; package $AUX_PACKAGE end $AUX_PACKAGE;\")", parsed = false)

# Loop over the inheritance chain and transform each class
for model in chain
    aux_model = AUX_NAME_MAP[model]
    aux_name = split(aux_model, ".")[end]

    println("Transforming $model -> $aux_model")
    omc_call(omc, "copyClass($model, \"$aux_name\", $AUX_PACKAGE)")

    components = components_by_model[model]

    apply_replacements!(omc, model, aux_model, components, SLACK_COMPONENT)
    delete_connections!(omc, aux_model, components; global_targets = global_blacklist_names)
    delete_components!(omc, aux_model, components)
    add_init_models!(omc, model, aux_model, components, INIT_MODEL_BY_COMPONENT, SLACK_COMPONENT)
    apply_LF_modifiers!(omc, model, aux_model, components)
    add_init_equations!(omc, model, aux_model, components, INIT_MODEL_BY_COMPONENT, SLACK_COMPONENT)
end

# Write the auxiliary package to disk
isdir(AUX_DIR) && rm(AUX_DIR; recursive = true, force = true)
mkpath(AUX_DIR)

write_package_files!(
    AUX_PACKAGE_FILE,
    AUX_ORDER_FILE,
    AUX_PACKAGE,
    package_class_names(chain, AUX_NAME_MAP),
)

save_auxiliary_package_classes!(
    omc,
    chain,
    AUX_NAME_MAP,
    AUX_DIR,
    patch_components_by_model,
    SLACK_COMPONENT,
)

# Re-load the generated auxiliary package from disk and validate it
omc_call(omc, "deleteClass($AUX_PACKAGE)")
omc_call(omc, "loadFile(\"$AUX_PACKAGE_FILE\")")
chk = sendExpression(omc, "checkModel($AUX_ROOT_MODEL)", parsed=false)
println(chk)

println("Wrote auxiliary package: ", AUX_DIR)

Transforming MyIEEE14.IEEE14Base -> MyIEEE14_auxiliary.IEEE14Base_auxiliary


Transforming MyIEEE14.IEEE14DisconnectLine -> MyIEEE14_auxiliary.IEEE14DisconnectLine_auxiliary


"Check of MyIEEE14_auxiliary.IEEE14DisconnectLine_auxiliary completed successfully.
Class MyIEEE14_auxiliary.IEEE14DisconnectLine_auxiliary has 875 equation(s) and 875 variable(s).
326 of these are trivial equation(s)."

Wrote auxiliary package: /home/clarafercas/dynawo-notebooks/OpenModelica_only_users/BuildAux/MyIEEE14_auxiliary
